# vowel0: current SVM/KNN/RF/LR comparison

Methods: no resampling, SMOTE, BSMOTE, ROS, ADASYN, CASB, CASB-relaxed, CASB-edited, CASB-ROS. RUS, SMOTEENN and MCSMOTE are excluded. Historical records remain in metadata.

CASB uses its original interpolation generator; CASB-ROS duplicates sources selected with CASB's original 50/30/20 quotas. This modular extension is inspired by the UGO paper, not a reproduction of UGO. Five supplied folds, seeds 42/137, same training-only ratio search (0.5/0.75/1) maximizing (G-mean+F1)/2. RBF SVM C=1 gamma=scale; distance-weighted KNN k=min(7,resampled count). RF: 100 trees, minimum leaf size 2; LR: C=1, max_iter=3000. These classifiers were retained after inspecting benchmark results, so the expanded comparison is exploratory. All preprocessing is fitted on training data. Abalone uses one-hot encoding.

Run all cells to inspect cached evidence. RF/LR evidence is imported from progress.md; GaussianNB remains in that report. Use the shared data_splits, choose_ratio and evaluate_fit functions in binary_oversampling.py to refit any saved setting. ADASYN failures are reported without fallback; incomplete method averages are withheld. All datasets have been inspected during development; this is exploratory evidence.


In [1]:
import sys, os
from pathlib import Path
sys.dont_write_bytecode=True
sys.path.insert(0,str(Path(os.environ.get('TEMP','/tmp'))/'ml_project_runtime'))
sys.path.insert(0,str(Path.cwd()))
import pandas as pd
from IPython.display import display, Markdown
from binary_oversampling import current_dataset_records, CURRENT_METHODS
import json
nb=json.loads(Path('05_vowel0_CASB.ipynb').read_text(encoding='utf8'))
records=nb['metadata']['current_comparison']['records']
classifiers=['svm', 'knn', 'rf', 'lr']
frame=pd.DataFrame(records)
assert len(frame)==360
if 'error' not in frame:frame['error']=None
errors=frame[frame.error.notna()]
print('Attempted:',len(frame),'Failed:',len(errors))
if len(errors):display(errors[['classifier','method','seed','fold','error']])
frame['composite']=(frame.gmean+frame.f1)/2
metrics=['gmean','f1','precision','recall','specificity','ba','ap','auc','composite']


Attempted: 360 Failed: 0


In [2]:
ok=frame[frame.error.isna()]
counts=ok.groupby(['classifier','method']).size()
summary=ok.groupby(['classifier','method'])[metrics].mean()
for key in summary.index:
    if counts.loc[key]!=10:summary.loc[key,:]=float('nan')
summary=summary.reindex(pd.MultiIndex.from_product([classifiers,CURRENT_METHODS],names=['classifier','method']))
for clf in classifiers:
    display(Markdown('## '+clf.upper()+': averages (%)'))
    display((100*summary.loc[clf]).round(3))
combined=summary.groupby('method').mean().reindex(CURRENT_METHODS)
for method in CURRENT_METHODS:
    if summary.xs(method,level='method').isna().any().any():combined.loc[method,:]=float('nan')
display(Markdown('## Equal average of SVM, KNN, RF and LR (%)'))
display((100*combined).round(3))
print('NaN indicates incomplete coverage, not a zero score.')


## SVM: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,98.257,98.182,100.0,96.667,100.0,98.333,100.0,100.0,98.22
smote,100.000,100.000,100.0,100.000,100.0,100.000,100.0,100.0,100.00
bsmote,100.000,100.000,100.0,100.000,100.0,100.000,100.0,100.0,100.00
ros,100.000,100.000,100.0,100.000,100.0,100.000,100.0,100.0,100.00
adasyn,100.000,100.000,100.0,100.000,100.0,100.000,100.0,100.0,100.00
casb,100.000,100.000,100.0,100.000,100.0,100.000,100.0,100.0,100.00
casb_relaxed,100.000,100.000,100.0,100.000,100.0,100.000,100.0,100.0,100.00
casb_edited,100.000,100.000,100.0,100.000,100.0,100.000,100.0,100.0,100.00
casb_ros,100.000,100.000,100.0,100.000,100.0,100.000,100.0,100.0,100.00


## KNN: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,99.381,98.888,98.947,98.889,99.889,99.389,100.000,100.000,99.134
smote,99.944,99.459,98.947,100.000,99.889,99.944,100.000,100.000,99.702
bsmote,99.944,99.459,98.947,100.000,99.889,99.944,99.942,99.994,99.702
ros,99.944,99.459,98.947,100.000,99.889,99.944,100.000,100.000,99.702
adasyn,99.944,99.459,98.947,100.000,99.889,99.944,99.942,99.994,99.702
casb,99.944,99.459,98.947,100.000,99.889,99.944,100.000,100.000,99.702
casb_relaxed,99.944,99.459,98.947,100.000,99.889,99.944,100.000,100.000,99.702
casb_edited,99.944,99.459,98.947,100.000,99.889,99.944,100.000,100.000,99.702
casb_ros,99.944,99.459,98.947,100.000,99.889,99.944,100.000,100.000,99.702


## RF: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,94.262,92.958,97.680,89.444,99.778,94.611,98.959,99.867,93.610
smote,96.556,95.314,97.707,93.889,99.777,96.833,99.039,99.901,95.935
bsmote,96.247,94.773,97.204,93.333,99.721,96.527,99.457,99.944,95.510
ros,96.377,95.662,98.947,93.333,99.888,96.611,99.172,99.904,96.020
adasyn,96.833,95.175,96.707,94.444,99.665,97.055,98.954,99.879,96.004
casb,96.885,95.693,97.895,94.444,99.777,97.111,99.076,99.901,96.289
casb_relaxed,96.552,95.306,97.895,93.889,99.777,96.833,99.330,99.929,95.929
casb_edited,96.885,95.693,97.895,94.444,99.777,97.111,99.304,99.926,96.289
casb_ros,96.608,95.416,97.678,93.889,99.777,96.833,99.150,99.910,96.012


## LR: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,87.029,82.475,89.587,76.667,99.109,87.888,92.880,99.109,84.752
smote,95.259,81.522,72.289,94.444,96.216,95.330,93.245,99.316,88.390
bsmote,96.418,82.856,73.127,96.667,96.215,96.441,93.135,99.344,89.637
ros,94.751,81.471,73.039,93.333,96.328,94.831,93.155,99.319,88.111
adasyn,97.449,84.876,75.051,98.333,96.605,97.469,92.829,99.338,91.162
casb,95.654,82.538,73.616,95.000,96.383,95.691,92.551,99.304,89.096
casb_relaxed,96.079,82.064,72.119,96.111,96.104,96.108,92.644,99.295,89.072
casb_edited,96.338,82.191,71.995,96.667,96.048,96.358,92.714,99.304,89.264
casb_ros,95.771,81.706,72.024,95.556,96.049,95.802,93.087,99.329,88.739


## Equal average of SVM, KNN, RF and LR (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,94.732,93.126,96.554,90.417,99.694,95.055,97.960,99.744,93.929
smote,97.940,94.074,92.236,97.083,98.970,98.027,98.071,99.804,96.007
bsmote,98.152,94.272,92.320,97.500,98.956,98.228,98.133,99.821,96.212
ros,97.768,94.148,92.733,96.667,99.026,97.846,98.082,99.806,95.958
adasyn,98.557,94.878,92.676,98.194,99.040,98.617,97.931,99.803,96.717
casb,98.121,94.423,92.615,97.361,99.012,98.187,97.907,99.801,96.272
casb_relaxed,98.144,94.207,92.240,97.500,98.943,98.221,97.993,99.806,96.176
casb_edited,98.292,94.336,92.209,97.778,98.929,98.353,98.004,99.807,96.314
casb_ros,98.081,94.145,92.162,97.361,98.929,98.145,98.059,99.810,96.113


NaN indicates incomplete coverage, not a zero score.


In [3]:
core=['smote','bsmote','ros','adasyn']
family=['casb','casb_relaxed','casb_edited','casb_ros']
for view,table in [(clf,summary.loc[clf]) for clf in classifiers]+[('equal four-classifier average',combined)]:
    output=[]
    for metric in ['gmean','f1','precision','recall']:
        values=table.loc[core+family,metric]
        if values.isna().any():
            output.append(dict(metric=metric,result='Incomplete coverage'));continue
        ours=values.loc[family];base=values.loc[core];delta=ours.max()-base.max()
        output.append(dict(metric=metric,best_variant=ours.idxmax(),family_percent=100*ours.max(),baseline=base.idxmax(),baseline_percent=100*base.max(),difference_pp=100*delta,result='WIN' if delta>1e-12 else 'LOSS' if delta < -1e-12 else 'TIE'))
    display(Markdown('## '+view+': per-metric family comparison'))
    display(pd.DataFrame(output).round(3))
print('Retrospective per-metric family maxima; not one deployable winning method. No composite winner.')
display(Markdown('## Fold-level evidence'))
with pd.option_context('display.max_rows',400):
    display(frame[['seed','fold','classifier','method','ratio']+metrics+['error']])


## svm: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb,100.0,smote,100.0,0.0,TIE
1,f1,casb,100.0,smote,100.0,0.0,TIE
2,precision,casb,100.0,smote,100.0,0.0,TIE
3,recall,casb,100.0,smote,100.0,0.0,TIE


## knn: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb,99.944,smote,99.944,0.0,TIE
1,f1,casb,99.459,smote,99.459,0.0,TIE
2,precision,casb,98.947,smote,98.947,0.0,TIE
3,recall,casb,100.000,smote,100.000,0.0,TIE


## rf: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb,96.885,adasyn,96.833,0.052,WIN
1,f1,casb,95.693,ros,95.662,0.031,WIN
2,precision,casb,97.895,ros,98.947,-1.053,LOSS
3,recall,casb,94.444,adasyn,94.444,0.000,TIE


... (3 outputs left out)

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,98.292,adasyn,98.557,-0.265,LOSS
1,f1,casb,94.423,adasyn,94.878,-0.455,LOSS
2,precision,casb,92.615,ros,92.733,-0.119,LOSS
3,recall,casb_edited,97.778,adasyn,98.194,-0.417,LOSS


Retrospective per-metric family maxima; not one deployable winning method. No composite winner.


## Fold-level evidence

,seed,fold,classifier,method,ratio,gmean,f1,precision,recall,specificity,ba,ap,auc,composite,error
0,42,1,svm,original,0.50,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,None
1,42,1,svm,smote,0.75,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,None
2,42,1,svm,bsmote,0.50,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,None
3,42,1,svm,ros,0.50,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,None
4,42,1,svm,adasyn,0.50,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,None
5,42,1,svm,casb,0.50,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,None
6,42,1,svm,casb_relaxed,1.00,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,None
7,42,1,svm,casb_edited,1.00,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,None
8,42,1,svm,casb_ros,0.50,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,None
9,42,1,knn,original,0.50,0.997218,0.972973,0.947368,1.000000,0.994444,0.997222,1.000000,1.000000,0.985096,None
